
> **v2 GPU-safe profile:** tuned for RTX 4050 Laptop 6GB with `BATCH_SIZE=4`, AMP on CUDA, sequential two-view forward, and a 2-sample sanity pass. Existing mouth cache is reused; no preprocessing is repeated.

# VocaLip Thai — Notebook 02 v2
## Self-Supervised Visual Pretraining (SSL)

**Goal:** pretrain the mouth-video encoder without using word labels in the loss, then transfer the encoder to Notebook 03 for 44-word supervised fine-tuning.

### Experiment design
- Input: cached mouth ROI clips, `(16, 112, 112, 3)`
- SSL data:
  - **train split** → SSL optimization
  - **val split** → SSL validation only
  - **test split is not used**
- Two stochastic views of the same mouth clip
- Clip encoder:
  - FrameCNN
  - 2-layer BiGRU
  - Temporal Attention
- SSL objective: **NT-Xent / InfoNCE contrastive loss**
- Projection head used only during SSL
- Output:
  - best SSL checkpoint
  - encoder-only checkpoint
  - training history
  - experiment summary JSON

> Important: This notebook is a clean SSL ablation. It does not use the 44 class labels for optimization.


## Notebook flow

1. Imports / environment  
2. Config  
3. Reproducibility  
4. Load clean manifest  
5. Cache resolver  
6. SSL augmentations  
7. SSL dataset + DataLoaders  
8. Visual encoder  
9. Projection head + SSL model  
10. NT-Xent loss  
11. Train/eval helpers  
12. Sanity forward pass  
13. SSL pretraining  
14. Load best SSL checkpoint  
15. Training curves  
16. Embedding diagnostics  
17. Export encoder for Notebook 03  
18. Final experiment record  


In [ ]:
# ============================================================
# CELL 1 : IMPORTS / ENVIRONMENT
# ============================================================

import os
import sys
import json
import math
import time
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

print("=" * 80)
print("VOCALIP THAI — NOTEBOOK 02 / SSL PRETRAINING")
print("=" * 80)
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("CUDA   :", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = DEVICE

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

print("Device :", DEVICE)
print("scikit-learn required: NO")


In [ ]:
# ============================================================
# CELL 2 : CONFIG
# ============================================================

ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")

MANIFEST_PATH = ROOT / "word_train_manifest_v5_baseline_clean.csv"
CACHE_ROOT = ROOT / "mouth_cache_v3"

SSL_DIR = ROOT / "models_02_ssl_v2"
SSL_DIR.mkdir(parents=True, exist_ok=True)

SSL_BEST_PATH = SSL_DIR / "02_ssl_v2_best.pt"
SSL_LAST_PATH = SSL_DIR / "02_ssl_v2_last.pt"
SSL_ENCODER_PATH = SSL_DIR / "02_ssl_v2_encoder_for_finetuning.pt"
SSL_HISTORY_PATH = SSL_DIR / "02_ssl_v2_history.csv"
SSL_SUMMARY_PATH = SSL_DIR / "02_ssl_v2_summary.json"

NUM_CLASSES = 44

TARGET_FRAMES = 16
ROI_SIZE = 112
CHANNELS = 3

# Keep encoder dimensions aligned with Experiment 0
FEATURE_DIM = 128
GRU_HIDDEN = 256
GRU_LAYERS = 2
ENCODER_DROPOUT = 0.20
FRAME_PROJECTION_DROPOUT = 0.10

# SSL projection head
PROJECTION_DIM = 128
PROJECTION_HIDDEN = 256

# Optimization
BATCH_SIZE = 4
NUM_WORKERS = 0
EPOCHS = 50
LR = 3e-4
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.20
GRAD_CLIP_NORM = 5.0
PATIENCE = 8

# Scheduler
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3
SCHEDULER_MIN_LR = 1e-6

# SSL augmentation strength — intentionally mild
BRIGHTNESS_PROB = 0.60
BRIGHTNESS_RANGE = (0.85, 1.15)

CONTRAST_PROB = 0.60
CONTRAST_RANGE = (0.85, 1.15)

NOISE_PROB = 0.30
NOISE_STD_MAX = 0.025

BLUR_PROB = 0.20
TEMPORAL_SHIFT_PROB = 0.40
TEMPORAL_SHIFT_MAX = 1

FRAME_MASK_PROB = 0.20
FRAME_MASK_MAX = 2

# Intentionally disabled:
USE_HORIZONTAL_FLIP = False
USE_TEMPORAL_REVERSE = False
USE_HEAVY_GEOMETRIC_AUG = False

USE_AMP = bool(torch.cuda.is_available())

print("=" * 80)
print("CONFIG")
print("=" * 80)
print("Manifest :", MANIFEST_PATH)
print("Cache    :", CACHE_ROOT)
print("SSL dir  :", SSL_DIR)
print("Frames   :", TARGET_FRAMES)
print("ROI      :", ROI_SIZE)
print("Batch    :", BATCH_SIZE)
print("Epochs   :", EPOCHS)
print("Temp     :", TEMPERATURE)
print("Device   :", DEVICE)
print("AMP      :", USE_AMP)

assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"
assert CACHE_ROOT.exists(), f"Missing cache root: {CACHE_ROOT}"


In [ ]:
# ============================================================
# CELL 3 : REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


In [ ]:
# ============================================================
# CELL 4 : LOAD CLEAN MANIFEST
# ============================================================

df = pd.read_csv(MANIFEST_PATH)

required_columns = {"split", "model_class_id", "word"}
missing = required_columns - set(df.columns)

assert not missing, f"Missing columns: {missing}"

df["model_class_id"] = df["model_class_id"].astype(int)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print("=" * 80)
print("DATASET")
print("=" * 80)
print("Total :", len(df))
print("Train :", len(train_df))
print("Val   :", len(val_df))
print("Test  :", len(test_df))
print("Classes:", df["model_class_id"].nunique())
print()
print("SSL policy:")
print("  train -> optimization")
print("  val   -> SSL validation")
print("  test  -> NOT USED")

assert len(df) == 2245
assert len(train_df) == 1632
assert len(val_df) == 354
assert len(test_df) == 259
assert df["model_class_id"].nunique() == 44


In [ ]:
# ============================================================
# CELL 5 : CACHE PATH RESOLVER
# ============================================================

def resolve_cache_path(row):
    # Prefer explicit cached path if present.
    for key in ["mouth_cache_path", "cache_path"]:
        if key in row.index and pd.notna(row[key]):
            p = Path(str(row[key]))
            if p.exists():
                return p

    # Identifier-based fallbacks.
    for key in [
        "clip_id",
        "segment_id",
        "sample_id",
        "video_id",
        "filename",
        "clip_name",
    ]:
        if key in row.index and pd.notna(row[key]):
            stem = Path(str(row[key])).stem

            candidates = [
                CACHE_ROOT / f"{stem}.npz",
                CACHE_ROOT / stem / "mouth.npz",
            ]

            for p in candidates:
                if p.exists():
                    return p

    if "video_path" in row.index and pd.notna(row["video_path"]):
        stem = Path(str(row["video_path"])).stem
        p = CACHE_ROOT / f"{stem}.npz"
        if p.exists():
            return p

    raise FileNotFoundError(
        "Cannot resolve cache path for row:\n"
        + str(row.to_dict())
    )


# Fast audit before SSL training
for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
]:
    missing_paths = []

    for i in range(len(split_df)):
        try:
            _ = resolve_cache_path(split_df.iloc[i])
        except Exception:
            missing_paths.append(i)

    print(
        f"{split_name}: "
        f"{len(split_df) - len(missing_paths)}/{len(split_df)} cache files resolved"
    )

    assert len(missing_paths) == 0, (
        f"{split_name} has unresolved cache rows: "
        f"{missing_paths[:10]}"
    )

print("✅ SSL cache audit passed.")


In [ ]:
# ============================================================
# CELL 6 : SSL AUGMENTATIONS
# ============================================================

IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32
).view(1, 3, 1, 1)


def temporal_shift(frames, max_shift=1):
    if max_shift <= 0:
        return frames

    shift = random.randint(-max_shift, max_shift)

    if shift == 0:
        return frames

    if shift > 0:
        left = frames[:1].repeat(shift, 1, 1, 1)
        return torch.cat([left, frames[:-shift]], dim=0)

    shift = abs(shift)
    right = frames[-1:].repeat(shift, 1, 1, 1)
    return torch.cat([frames[shift:], right], dim=0)


def mild_blur(frames):
    # 3x3 average blur implemented without torchvision/OpenCV dependency.
    T, C, H, W = frames.shape
    x = frames.reshape(T * C, 1, H, W)
    x = F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)
    return x.reshape(T, C, H, W)


def mask_random_frames(frames, max_frames=2):
    if max_frames <= 0:
        return frames

    out = frames.clone()
    n = random.randint(1, max_frames)
    ids = random.sample(range(out.shape[0]), k=min(n, out.shape[0]))

    # Replace selected frame by nearest temporal neighbour, not zeros.
    for idx in ids:
        if idx == 0:
            src = 1
        else:
            src = idx - 1
        out[idx] = out[src]

    return out


def make_ssl_view(frames_uint8):
    # Input: numpy uint8 T,H,W,C
    x = torch.from_numpy(frames_uint8.copy()).float() / 255.0
    x = x.permute(0, 3, 1, 2)  # T,C,H,W

    if random.random() < BRIGHTNESS_PROB:
        factor = random.uniform(*BRIGHTNESS_RANGE)
        x = x * factor

    if random.random() < CONTRAST_PROB:
        factor = random.uniform(*CONTRAST_RANGE)
        mean = x.mean(dim=(2, 3), keepdim=True)
        x = (x - mean) * factor + mean

    if random.random() < NOISE_PROB:
        std = random.uniform(0.005, NOISE_STD_MAX)
        x = x + torch.randn_like(x) * std

    if random.random() < BLUR_PROB:
        x = mild_blur(x)

    if random.random() < TEMPORAL_SHIFT_PROB:
        x = temporal_shift(
            x,
            max_shift=TEMPORAL_SHIFT_MAX
        )

    if random.random() < FRAME_MASK_PROB:
        x = mask_random_frames(
            x,
            max_frames=FRAME_MASK_MAX
        )

    x = torch.clamp(x, 0.0, 1.0)

    # No horizontal flip / no temporal reverse.
    x = (x - IMAGENET_MEAN) / IMAGENET_STD

    return x


print("SSL augmentation pipeline: READY")
print("Horizontal flip :", USE_HORIZONTAL_FLIP)
print("Temporal reverse:", USE_TEMPORAL_REVERSE)
print("Heavy geometry  :", USE_HEAVY_GEOMETRIC_AUG)


In [ ]:
# ============================================================
# CELL 7 : SSL DATASET + DATALOADERS
# ============================================================

class VocaLipSSLDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cache_path = resolve_cache_path(row)

        with np.load(cache_path) as data:
            frames = data["frames"].copy()

        assert frames.shape == (
            TARGET_FRAMES,
            ROI_SIZE,
            ROI_SIZE,
            3
        ), f"Bad cache shape {frames.shape}: {cache_path}"

        view1 = make_ssl_view(frames)
        view2 = make_ssl_view(frames)

        # Returned only for diagnostics; NOT used by SSL loss.
        class_id = int(row["model_class_id"])

        return view1, view2, class_id


ssl_train_dataset = VocaLipSSLDataset(train_df)
ssl_val_dataset   = VocaLipSSLDataset(val_df)

pin_memory = torch.cuda.is_available()

ssl_train_loader = DataLoader(
    ssl_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,   # stable NT-Xent batch geometry
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

ssl_val_loader = DataLoader(
    ssl_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

v1, v2, y = ssl_train_dataset[0]

print("=" * 80)
print("SSL DATASET")
print("=" * 80)
print("Train clips :", len(ssl_train_dataset))
print("Val clips   :", len(ssl_val_dataset))
print("View1 shape :", tuple(v1.shape))
print("View2 shape :", tuple(v2.shape))
print("Diagnostic label:", y)
print("Train batches:", len(ssl_train_loader))
print("Val batches  :", len(ssl_val_loader))

assert v1.shape == (TARGET_FRAMES, 3, ROI_SIZE, ROI_SIZE)
assert v2.shape == (TARGET_FRAMES, 3, ROI_SIZE, ROI_SIZE)


In [ ]:
# ============================================================
# CELL 8 : VISUAL ENCODER
# Same backbone family as Experiment 0
# ============================================================

class FrameCNNEncoder(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 96, 3, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(96, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=FRAME_PROJECTION_DROPOUT),
        )

    def forward(self, x):
        return self.projection(self.features(x))


class TemporalAttention(nn.Module):
    def __init__(self, input_dim, attention_dim=128):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1),
        )

    def forward(self, x):
        scores = self.attention(x).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )
        return pooled, weights


class VisualClipEncoder(nn.Module):
    def __init__(
        self,
        feature_dim=128,
        gru_hidden=256,
        gru_layers=2,
        dropout=0.20,
    ):
        super().__init__()

        self.frame_encoder = FrameCNNEncoder(
            feature_dim=feature_dim
        )

        self.temporal_encoder = nn.GRU(
            input_size=feature_dim,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=(dropout if gru_layers > 1 else 0.0),
        )

        self.output_dim = gru_hidden * 2

        self.temporal_attention = TemporalAttention(
            input_dim=self.output_dim,
            attention_dim=128,
        )

    def forward(self, video, return_attention=False):
        B, T, C, H, W = video.shape

        x = video.reshape(B * T, C, H, W)
        frame_features = self.frame_encoder(x)
        frame_features = frame_features.reshape(B, T, -1)

        temporal_features, _ = self.temporal_encoder(
            frame_features
        )

        pooled, attention_weights = self.temporal_attention(
            temporal_features
        )

        if return_attention:
            return pooled, attention_weights

        return pooled


encoder_probe = VisualClipEncoder(
    feature_dim=FEATURE_DIM,
    gru_hidden=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=ENCODER_DROPOUT,
)

print("Encoder output dim:", encoder_probe.output_dim)
print(
    "Encoder parameters:",
    f"{sum(p.numel() for p in encoder_probe.parameters()):,}"
)


In [ ]:
# ============================================================
# CELL 9 : SSL MODEL + PROJECTION HEAD
# ============================================================

class SSLProjectionHead(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        output_dim=128
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        z = self.net(x)
        return F.normalize(z, dim=1)


class VocaLipSSLModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = VisualClipEncoder(
            feature_dim=FEATURE_DIM,
            gru_hidden=GRU_HIDDEN,
            gru_layers=GRU_LAYERS,
            dropout=ENCODER_DROPOUT,
        )

        self.projector = SSLProjectionHead(
            input_dim=self.encoder.output_dim,
            hidden_dim=PROJECTION_HIDDEN,
            output_dim=PROJECTION_DIM,
        )

    def forward(self, video, return_embedding=False):
        embedding, attention = self.encoder(
            video,
            return_attention=True
        )

        projection = self.projector(embedding)

        if return_embedding:
            return projection, embedding, attention

        return projection


ssl_model = VocaLipSSLModel().to(DEVICE)

ssl_total_params = sum(
    p.numel()
    for p in ssl_model.parameters()
)

print("=" * 80)
print("SSL MODEL")
print("=" * 80)
print("Encoder dim    :", ssl_model.encoder.output_dim)
print("Projection dim :", PROJECTION_DIM)
print("Parameters     :", f"{ssl_total_params:,}")


In [ ]:
# ============================================================
# CELL 10 : NT-XENT / INFO-NCE LOSS
# ============================================================

def nt_xent_loss(z1, z2, temperature=0.20):
    if z1.ndim != 2 or z2.ndim != 2:
        raise ValueError("z1 and z2 must be 2D tensors")

    if z1.shape != z2.shape:
        raise ValueError(
            f"Shape mismatch: {z1.shape} vs {z2.shape}"
        )

    batch_size = z1.size(0)

    if batch_size < 2:
        raise ValueError(
            "NT-Xent requires batch size >= 2"
        )

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    z = torch.cat([z1, z2], dim=0)   # 2B,D

    similarity = torch.matmul(z, z.T) / temperature

    # Remove self-similarity.
    diag_mask = torch.eye(
        2 * batch_size,
        dtype=torch.bool,
        device=z.device
    )

    similarity = similarity.masked_fill(
        diag_mask,
        float("-inf")
    )

    # Positive partner:
    # 0..B-1 <-> B..2B-1
    targets = torch.arange(
        2 * batch_size,
        device=z.device
    )

    targets = (targets + batch_size) % (2 * batch_size)

    loss = F.cross_entropy(
        similarity,
        targets
    )

    return loss


# quick numerical sanity
z1 = F.normalize(
    torch.randn(8, PROJECTION_DIM, device=DEVICE),
    dim=1
)
z2 = F.normalize(
    torch.randn(8, PROJECTION_DIM, device=DEVICE),
    dim=1
)

print(
    "Random NT-Xent:",
    float(nt_xent_loss(z1, z2, TEMPERATURE))
)


In [ ]:
# ============================================================
# CELL 11 : OPTIMIZER / SCHEDULER / AMP
# ============================================================

optimizer = torch.optim.AdamW(
    ssl_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    min_lr=SCHEDULER_MIN_LR,
)

if torch.cuda.is_available():
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
else:
    scaler = torch.amp.GradScaler("cpu", enabled=False)

print("Optimizer : AdamW")
print("LR        :", LR)
print("WD        :", WEIGHT_DECAY)
print("Scheduler : ReduceLROnPlateau / val SSL loss")
print("AMP       :", USE_AMP)


In [ ]:
# ============================================================
# CELL 12 : TRAIN / EVAL HELPERS
# ============================================================

def run_ssl_epoch(
    model,
    loader,
    optimizer=None,
    scaler=None,
    training=True,
):
    if training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    total_samples = 0

    iterator = tqdm(
        loader,
        desc="SSL train" if training else "SSL val",
        leave=False
    )

    for view1, view2, _diagnostic_label in iterator:
        view1 = view1.to(
            DEVICE,
            non_blocking=True
        )
        view2 = view2.to(
            DEVICE,
            non_blocking=True
        )

        batch_size = view1.size(0)

        if training:
            optimizer.zero_grad(set_to_none=True)

        grad_context = (
            torch.enable_grad()
            if training
            else torch.inference_mode()
        )

        with grad_context:
            if DEVICE.type == "cuda":
                autocast_ctx = torch.amp.autocast(
                    device_type="cuda",
                    enabled=USE_AMP
                )
            else:
                autocast_ctx = torch.amp.autocast(
                    device_type="cpu",
                    enabled=False
                )

            with autocast_ctx:
                z1 = model(view1)
                z2 = model(view2)

                loss = nt_xent_loss(
                    z1,
                    z2,
                    temperature=TEMPERATURE
                )

            if training:
                scaler.scale(loss).backward()

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP_NORM
                )

                scaler.step(optimizer)
                scaler.update()

        running_loss += float(loss.item()) * batch_size
        total_samples += batch_size

        iterator.set_postfix(
            loss=f"{running_loss / max(total_samples, 1):.4f}"
        )

    return running_loss / total_samples


print("Train/eval helpers: READY")


In [ ]:
# ============================================================
# CELL 13 : SANITY FORWARD PASS
# ============================================================

view1, view2, labels = next(
    iter(ssl_train_loader)
)

view1 = view1[:2].to(DEVICE)
view2 = view2[:2].to(DEVICE)
labels = labels[:2]

ssl_model.eval()

with torch.inference_mode():
    z1, emb1, attn1 = ssl_model(
        view1,
        return_embedding=True
    )
    z2, emb2, attn2 = ssl_model(
        view2,
        return_embedding=True
    )

    sanity_loss = nt_xent_loss(
        z1,
        z2,
        temperature=TEMPERATURE
    )

positive_similarity = F.cosine_similarity(
    z1,
    z2,
    dim=1
).mean().item()

print("=" * 80)
print("SANITY FORWARD")
print("=" * 80)
print("Input        :", tuple(view1.shape))
print("Embedding    :", tuple(emb1.shape))
print("Projection   :", tuple(z1.shape))
print("Attention    :", tuple(attn1.shape))
print("NT-Xent loss :", f"{float(sanity_loss):.4f}")
print("Pos cosine   :", f"{positive_similarity:.4f}")
print()
print("IMPORTANT: diagnostic word labels are NOT passed into SSL loss.")


In [ ]:
# ============================================================
# CELL 14 : SSL PRETRAINING
# ============================================================

history = []

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

training_start = time.time()

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.time()

    train_loss = run_ssl_epoch(
        model=ssl_model,
        loader=ssl_train_loader,
        optimizer=optimizer,
        scaler=scaler,
        training=True,
    )

    val_loss = run_ssl_epoch(
        model=ssl_model,
        loader=ssl_val_loader,
        optimizer=None,
        scaler=None,
        training=False,
    )

    current_lr = optimizer.param_groups[0]["lr"]

    scheduler.step(val_loss)

    epoch_seconds = time.time() - epoch_start

    row = {
        "epoch": epoch,
        "train_ssl_loss": float(train_loss),
        "val_ssl_loss": float(val_loss),
        "lr": float(current_lr),
        "seconds": float(epoch_seconds),
    }

    history.append(row)

    improved = val_loss < best_val_loss - 1e-6

    if improved:
        best_val_loss = float(val_loss)
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": ssl_model.state_dict(),
            "encoder_state_dict": ssl_model.encoder.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_ssl_loss": float(train_loss),
            "val_ssl_loss": float(val_loss),
            "config": {
                "feature_dim": FEATURE_DIM,
                "gru_hidden": GRU_HIDDEN,
                "gru_layers": GRU_LAYERS,
                "encoder_dropout": ENCODER_DROPOUT,
                "projection_dim": PROJECTION_DIM,
                "temperature": TEMPERATURE,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
            },
        }, SSL_BEST_PATH)

    else:
        epochs_without_improvement += 1

    # Always save last state
    torch.save({
        "epoch": epoch,
        "model_state_dict": ssl_model.state_dict(),
        "encoder_state_dict": ssl_model.encoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_ssl_loss": float(train_loss),
        "val_ssl_loss": float(val_loss),
    }, SSL_LAST_PATH)

    pd.DataFrame(history).to_csv(
        SSL_HISTORY_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train {train_loss:.4f} | "
        f"val {val_loss:.4f} | "
        f"best {best_val_loss:.4f} @ {best_epoch} | "
        f"lr {current_lr:.2e} | "
        f"{epoch_seconds:.1f}s"
    )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"Early stopping: no val SSL loss improvement "
            f"for {PATIENCE} epochs."
        )
        break

training_seconds = time.time() - training_start

print("=" * 80)
print("SSL PRETRAINING COMPLETE")
print("=" * 80)
print("Best epoch   :", best_epoch)
print("Best val loss:", f"{best_val_loss:.6f}")
print("Training min :", f"{training_seconds / 60:.2f}")
print("Best ckpt    :", SSL_BEST_PATH)
print("History      :", SSL_HISTORY_PATH)


In [ ]:
# ============================================================
# CELL 15 : RESTORE BEST SSL CHECKPOINT
# ============================================================

try:
    ssl_checkpoint = torch.load(
        SSL_BEST_PATH,
        map_location=DEVICE,
        weights_only=False
    )
except TypeError:
    ssl_checkpoint = torch.load(
        SSL_BEST_PATH,
        map_location=DEVICE
    )

ssl_model.load_state_dict(
    ssl_checkpoint["model_state_dict"],
    strict=True
)

ssl_model.eval()

print("=" * 80)
print("BEST SSL CHECKPOINT RESTORED")
print("=" * 80)
print("Epoch        :", ssl_checkpoint.get("epoch"))
print("Train loss   :", ssl_checkpoint.get("train_ssl_loss"))
print("Val loss     :", ssl_checkpoint.get("val_ssl_loss"))
print("Strict load  : OK")


In [ ]:
# ============================================================
# CELL 16 : TRAINING CURVES
# ============================================================

history_df = pd.read_csv(SSL_HISTORY_PATH)

display(history_df.tail(10))

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    history_df["epoch"],
    history_df["train_ssl_loss"],
    label="Train SSL loss"
)

ax.plot(
    history_df["epoch"],
    history_df["val_ssl_loss"],
    label="Val SSL loss"
)

ax.axvline(
    int(ssl_checkpoint["epoch"]),
    linestyle="--",
    label=f"Best epoch {ssl_checkpoint['epoch']}"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("NT-Xent loss")
ax.set_title("VocaLip Thai — SSL Pretraining Loss")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 17 : EMBEDDING DIAGNOSTICS
# No supervised training — labels only used for reporting.
# ============================================================

@torch.inference_mode()
def collect_embeddings(model, loader, max_batches=None):
    model.eval()

    embeddings = []
    labels = []

    for batch_idx, (v1, _v2, y) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        v1 = v1.to(DEVICE, non_blocking=True)

        _z, emb, _attn = model(
            v1,
            return_embedding=True
        )

        embeddings.append(emb.cpu())
        labels.append(y.cpu())

    return (
        torch.cat(embeddings).numpy(),
        torch.cat(labels).numpy()
    )


val_embeddings, val_labels = collect_embeddings(
    ssl_model,
    ssl_val_loader
)

embedding_norms = np.linalg.norm(
    val_embeddings,
    axis=1
)

print("=" * 80)
print("EMBEDDING DIAGNOSTICS")
print("=" * 80)
print("Shape        :", val_embeddings.shape)
print("Finite       :", np.isfinite(val_embeddings).all())
print("Mean norm    :", float(embedding_norms.mean()))
print("Std norm     :", float(embedding_norms.std()))
print("Mean abs val :", float(np.abs(val_embeddings).mean()))

assert np.isfinite(val_embeddings).all()
assert float(np.abs(val_embeddings).mean()) > 0.0

# Simple PCA using NumPy SVD for visualization only.
X = val_embeddings - val_embeddings.mean(axis=0, keepdims=True)
_, _, vt = np.linalg.svd(
    X,
    full_matrices=False
)
pca2 = X @ vt[:2].T

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    pca2[:, 0],
    pca2[:, 1],
    c=val_labels,
    s=18,
    alpha=0.75
)
ax.set_title(
    "SSL Validation Embeddings — PCA Projection\n"
    "(labels shown only for diagnostic visualization)"
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 18 : EXPORT ENCODER FOR NOTEBOOK 03
# ============================================================

encoder_export = {
    "source_experiment": "Notebook 02 / SSL Visual Pretraining",
    "best_ssl_epoch": int(ssl_checkpoint["epoch"]),
    "best_ssl_val_loss": float(ssl_checkpoint["val_ssl_loss"]),
    "encoder_state_dict": ssl_checkpoint["encoder_state_dict"],
    "encoder_config": {
        "target_frames": TARGET_FRAMES,
        "roi_size": ROI_SIZE,
        "feature_dim": FEATURE_DIM,
        "gru_hidden": GRU_HIDDEN,
        "gru_layers": GRU_LAYERS,
        "encoder_dropout": ENCODER_DROPOUT,
        "frame_projection_dropout": FRAME_PROJECTION_DROPOUT,
        "encoder_output_dim": GRU_HIDDEN * 2,
    },
    "pretraining": {
        "objective": "NT-Xent / InfoNCE",
        "temperature": TEMPERATURE,
        "train_split_only_for_optimization": True,
        "val_split_for_ssl_validation": True,
        "test_split_used": False,
        "horizontal_flip": False,
        "temporal_reverse": False,
    },
}

torch.save(
    encoder_export,
    SSL_ENCODER_PATH
)

print("=" * 80)
print("ENCODER EXPORTED")
print("=" * 80)
print("Path:", SSL_ENCODER_PATH)
print()
print("Notebook 03 can load:")
print("  checkpoint['encoder_state_dict']")
print("with strict=True into VisualClipEncoder.")


In [ ]:
# ============================================================
# CELL 19 : FINAL SSL EXPERIMENT RECORD
# ============================================================

def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)

    if not path.exists():
        return None

    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


ssl_record = {
    "project": "VocaLip Thai",
    "notebook": "02_vocalip_thai_ssl_visual_pretraining_v2",
    "experiment": "SSL visual pretraining",
    "status": "COMPLETED",

    "dataset": {
        "manifest": str(MANIFEST_PATH),
        "clean_total": int(len(df)),
        "ssl_train": int(len(train_df)),
        "ssl_val": int(len(val_df)),
        "test_excluded_from_ssl": int(len(test_df)),
        "classes_present": int(df["model_class_id"].nunique()),
    },

    "privacy_of_evaluation_split": {
        "word_labels_used_in_ssl_loss": False,
        "test_split_used_in_ssl_training": False,
        "test_split_used_in_ssl_validation": False,
    },

    "input": {
        "stream": "mouth-only",
        "frames": TARGET_FRAMES,
        "roi_size": ROI_SIZE,
        "normalization": "ImageNet mean/std",
    },

    "encoder": {
        "frame_encoder": "4-block CNN",
        "frame_feature_dim": FEATURE_DIM,
        "temporal_encoder": "2-layer bidirectional GRU",
        "gru_hidden": GRU_HIDDEN,
        "temporal_attention": True,
        "output_dim": GRU_HIDDEN * 2,
    },

    "ssl": {
        "objective": "NT-Xent / InfoNCE",
        "temperature": TEMPERATURE,
        "projection_hidden": PROJECTION_HIDDEN,
        "projection_dim": PROJECTION_DIM,
        "two_view_augmentation": True,
        "horizontal_flip": False,
        "temporal_reverse": False,
        "heavy_geometric_augmentation": False,
    },

    "optimization": {
        "optimizer": "AdamW",
        "initial_lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "planned_epochs": EPOCHS,
        "patience": PATIENCE,
        "grad_clip_norm": GRAD_CLIP_NORM,
        "amp": USE_AMP,
    },

    "result": {
        "best_epoch": int(ssl_checkpoint["epoch"]),
        "best_train_ssl_loss": float(
            ssl_checkpoint["train_ssl_loss"]
        ),
        "best_val_ssl_loss": float(
            ssl_checkpoint["val_ssl_loss"]
        ),
    },

    "artifacts": {
        "best_checkpoint": str(SSL_BEST_PATH),
        "last_checkpoint": str(SSL_LAST_PATH),
        "encoder_only": str(SSL_ENCODER_PATH),
        "history_csv": str(SSL_HISTORY_PATH),
    },

    "sha256": {
        "best_checkpoint": file_sha256(SSL_BEST_PATH),
        "encoder_only": file_sha256(SSL_ENCODER_PATH),
        "manifest": file_sha256(MANIFEST_PATH),
    },

    "next": (
        "Notebook 03 — SSL Fine-Tuning for 44 Thai Words; "
        "compare against frozen Experiment 0 baseline"
    ),
}

with open(
    SSL_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        ssl_record,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 88)
print("NOTEBOOK 02 — SSL FINAL RECORD")
print("=" * 88)
print("SSL train clips :", len(train_df))
print("SSL val clips   :", len(val_df))
print("Test used       : NO")
print("Labels in loss  : NO")
print("Best epoch      :", ssl_checkpoint["epoch"])
print(
    "Best val loss   :",
    f"{float(ssl_checkpoint['val_ssl_loss']):.6f}"
)
print("Encoder export  :", SSL_ENCODER_PATH)
print("Summary JSON    :", SSL_SUMMARY_PATH)
print()
print("NEXT: Notebook 03 — SSL Fine-Tuning for 44 Thai Words")
